# Official GPT Training on Google Colab

This notebook acts as an orchestration layer to seamlessly run the platform-independent GPT codebase on Google Colab without modifying the core logic.

**Note:** To enable Google Drive integration for persistent checkpoints, run the cell below.

In [ ]:
# 1. (Optional) Mount Google Drive for persistent checkpoints
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/GPT2-Run'
    print(f"Google Drive mounted! Training will save to {OUT_DIR}")
except ImportError:
    OUT_DIR = 'runs/colab_run'
    print(f"Not running in Colab (or Drive not mounted). Checkpoints will save to local {OUT_DIR}")

## 2. Setup the Environment

We need to clone the repository (if not already present) and install dependencies.

In [ ]:
# Install dependencies
!pip install -r requirements.txt

# Verify GPU
!nvidia-smi

## 3. Dataset Preparation

If you haven't prepared the Tiny Shakespeare dataset yet, run this cell to download and encode it into `.bin` files.

In [ ]:
!python scripts/prepare_data.py

## 4. Training (With Auto-Resume)

The training script automatically detects `CUDA`, `MPS`, or `CPU`.

By specifying `--out_dir {OUT_DIR}`, the script will automatically resume training from `latest.pt` if your Colab session reconnects!

In [ ]:
# Adjust hyperparameters based on your Colab GPU (e.g. increase batch_size on A100)
!python scripts/train.py \
    --train_bin data/train.bin \
    --val_bin data/val.bin \
    --out_dir {OUT_DIR} \
    --vocab_size 50257 \
    --d_model 256 \
    --n_heads 8 \
    --n_layers 6 \
    --context_length 256 \
    --batch_size 64 \
    --max_iters 5000 \
    --eval_interval 500 \
    --learning_rate 6e-4

## 5. Text Generation

Test your trained model!

In [ ]:
import os
best_checkpoint = os.path.join(OUT_DIR, "best.pt")

!python scripts/generate.py \
    --checkpoint {best_checkpoint} \
    --prompt "ROMEO:" \
    --max_new_tokens 200 \
    --temperature 0.8